# Production Patterns with Agenkit

Building production-ready AI agent systems requires robust patterns for reliability, observability, and performance.

## What You'll Learn

1. **Middleware** - Retry logic, circuit breakers, timeouts
2. **Observability** - Tracing and metrics
3. **Error Handling** - Fallback strategies
4. **Performance** - Caching and optimization
5. **Testing** - Unit and integration tests

## Prerequisites

- Completed Tutorial 01: Getting Started
- Understanding of async Python

> **Note**: For a more interactive experience with sliders and reactive execution, try the [Marimo version](02-production-patterns.py)!

Let's build production-grade systems! 🏗️

In [ ]:
# Setup
import agenkit
from agenkit import Agent, Message
import asyncio
import time

print(f"✅ Agenkit version: {agenkit.__version__}")

## 1. Middleware: Building Robust Agents

### Retry Middleware

Automatically retry failed operations with exponential backoff:

In [ ]:
from agenkit.middleware import RetryMiddleware


class UnreliableAgent(Agent):
    """Agent that fails first 2 attempts."""

    def __init__(self):
        self.attempt = 0

    def name(self) -> str:
        return "unreliable-agent"

    async def process(self, message: Message) -> Message:
        self.attempt += 1

        if self.attempt < 3:
            raise Exception(f"Simulated failure (attempt {self.attempt})")

        return Message(
            role="assistant",
            content=f"Success on attempt {self.attempt}!",
            metadata={"attempts": self.attempt},
        )


# Wrap with retry middleware
unreliable = UnreliableAgent()
retry_agent = RetryMiddleware(agent=unreliable, max_retries=3, backoff_factor=2.0)

print("✅ Created retry middleware")
print("   Max retries: 3")
print("   Backoff factor: 2.0x")

In [ ]:
# Test retry behavior
start = time.time()
result = await retry_agent.process(Message(role="user", content="test"))
duration = time.time() - start

print(f"✅ {result.content}")
print(f"   Total time: {duration:.2f}s")
print(f"   Attempts: {result.metadata['attempts']}")

### Circuit Breaker Middleware

Prevent cascading failures by opening the circuit after repeated failures:

In [ ]:
from agenkit.middleware import CircuitBreakerMiddleware

unreliable2 = UnreliableAgent()
circuit_breaker = CircuitBreakerMiddleware(
    agent=unreliable2, failure_threshold=3, recovery_timeout=10.0
)

print("✅ Created circuit breaker")
print("   Failure threshold: 3")
print("   Recovery timeout: 10s")

### Timeout Middleware

Prevent operations from hanging indefinitely:

In [ ]:
from agenkit.middleware import TimeoutMiddleware


class SlowAgent(Agent):
    def name(self) -> str:
        return "slow-agent"

    async def process(self, message: Message) -> Message:
        await asyncio.sleep(5)  # Simulate slow operation
        return Message(role="assistant", content="Done!")


slow = SlowAgent()
timeout_agent = TimeoutMiddleware(agent=slow, timeout=2.0)

# Test timeout
try:
    await timeout_agent.process(Message(role="user", content="test"))
except asyncio.TimeoutError:
    print("❌ Request timed out after 2 seconds (expected)")

## 2. Observability

### Tracing with OpenTelemetry

In [ ]:
from agenkit.observability import TracingMiddleware


class BusinessAgent(Agent):
    def name(self) -> str:
        return "business-agent"

    async def process(self, message: Message) -> Message:
        return Message(role="assistant", content=f"Processed: {message.content}")


business = BusinessAgent()
traced_agent = TracingMiddleware(agent=business, service_name="agenkit-tutorial")

print("✅ Created traced agent")
print("   Traces exported to configured backend")

### Metrics Collection

In [ ]:
from agenkit.observability import MetricsMiddleware


class MeteredAgent(Agent):
    def name(self) -> str:
        return "metered-agent"

    async def process(self, message: Message) -> Message:
        await asyncio.sleep(0.1)  # Simulate work
        return Message(role="assistant", content="Done")


metered = MeteredAgent()
metrics_agent = MetricsMiddleware(agent=metered, prefix="agenkit_", labels={"env": "tutorial"})

# Test metrics collection
result = await metrics_agent.process(Message(role="user", content="test"))
print(f"✅ {result.content}")
print("   Metrics exported with prefix: agenkit_")

## 3. Error Handling

### Fallback Pattern

In [ ]:
from agenkit.patterns import FallbackAgent

# Primary agent (unreliable)
primary = UnreliableAgent()


# Fallback agent (always works)
class SimpleAgent(Agent):
    def name(self) -> str:
        return "fallback-agent"

    async def process(self, message: Message) -> Message:
        return Message(
            role="assistant", content="Using fallback response", metadata={"fallback": True}
        )


fallback_simple = SimpleAgent()
fallback_chain = FallbackAgent(primary=primary, fallback=fallback_simple)

# Test fallback
result = await fallback_chain.process(Message(role="user", content="test"))
print(f"Response: {result.content}")
print(f"Used fallback: {result.metadata.get('fallback', False)}")

### Human-in-the-Loop

In [ ]:
from agenkit.patterns import HumanInLoopAgent


class CriticalAgent(Agent):
    def name(self) -> str:
        return "critical-agent"

    async def process(self, message: Message) -> Message:
        return Message(
            role="assistant", content="This is a critical decision", metadata={"confidence": 0.6}
        )


critical = CriticalAgent()
human_loop = HumanInLoopAgent(
    agent=critical,
    confidence_threshold=0.8,
    review_callback=lambda msg: print(f"⚠️  Human review needed: {msg.content}"),
)

print("✅ Created human-in-loop agent")
print("   Confidence threshold: 80%")

## 4. Performance Optimization

### Caching

In [ ]:
from agenkit.middleware import CachingMiddleware


class ExpensiveAgent(Agent):
    def __init__(self):
        self.call_count = 0

    def name(self) -> str:
        return "expensive-agent"

    async def process(self, message: Message) -> Message:
        self.call_count += 1
        await asyncio.sleep(1.0)  # Expensive operation

        return Message(role="assistant", content=f"Result (call #{self.call_count})")


expensive = ExpensiveAgent()
cached_agent = CachingMiddleware(agent=expensive, ttl=60.0)

msg = Message(role="user", content="same query")

# First call (cache miss)
start = time.time()
result1 = await cached_agent.process(msg)
duration1 = time.time() - start

# Second call (cache hit)
start = time.time()
result2 = await cached_agent.process(msg)
duration2 = time.time() - start

print(f"First call: {duration1:.2f}s - {result1.content}")
print(f"Second call: {duration2:.2f}s - {result2.content} (cached)")
print(f"Speedup: {duration1 / duration2:.1f}x faster 🚀")

## 5. Testing

### Unit Testing

In [ ]:
class TestAgent(Agent):
    def name(self) -> str:
        return "test-agent"

    async def process(self, message: Message) -> Message:
        return Message(role="assistant", content=f"Processed: {message.content}")


async def test_agent_basic():
    """Test basic agent functionality."""
    agent = TestAgent()

    # Test name
    assert agent.name() == "test-agent"
    print("✅ Test 1 passed: Correct name")

    # Test processing
    msg = Message(role="user", content="hello")
    result = await agent.process(msg)

    assert result.role == "assistant"
    print("✅ Test 2 passed: Correct role")

    assert "hello" in result.content
    print("✅ Test 3 passed: Content processed")

    return "All tests passed! 🎉"


test_result = await test_agent_basic()
print(f"\n{test_result}")

### Mock Agents

In [ ]:
class MockLLMAgent(Agent):
    """Mock LLM for testing without API calls."""

    def __init__(self, responses: dict[str, str]):
        self.responses = responses

    def name(self) -> str:
        return "mock-llm"

    async def process(self, message: Message) -> Message:
        response = self.responses.get(message.content, "Default mock response")
        return Message(role="assistant", content=response)


# Create mock with test data
mock_llm = MockLLMAgent(
    responses={
        "What is 2+2?": "4",
        "Hello": "Hi there!",
    }
)

# Test with mock
response = await mock_llm.process(Message(role="user", content="What is 2+2?"))
assert response.content == "4"
print("✅ Mock test passed")

## Summary

You've learned production-ready patterns:

✅ **Middleware** - Retry, circuit breaker, timeout  
✅ **Observability** - Tracing and metrics  
✅ **Error Handling** - Fallback and human-in-loop  
✅ **Performance** - Caching  
✅ **Testing** - Unit tests and mocks  

## Next Steps

- **[Tutorial 03: Advanced Reasoning](03-advanced-reasoning.ipynb)** - Chain-of-Thought, Tree-of-Thought
- **[Marimo Version](02-production-patterns.py)** - Interactive notebook with sliders and reactive execution
- **[Deployment Guide](../docs/deployment.md)** - Docker, Kubernetes
- **[Production Examples](https://github.com/scttfrdmn/agenkit/tree/main/examples/apps)** - Real-world applications

## Best Practices

1. Always use middleware in production
2. Add observability from day one
3. Test thoroughly with unit and integration tests
4. Plan for failures with fallbacks
5. Monitor and optimize performance

Ready to ship! 🚀